# 🎨 Lab 2: The Pixel Artist — Repaint Your Photo with K-Means!

Your phone photos contain **millions of different colors**. But what if we told K-Means: *"repaint this photo using only 8 colors"*? 🖌️

That's not just art — it's **image compression**, and it's clustering! Every pixel is a data point, its RGB values are the features, and K-Means groups similar colors together. **Same algorithm as the mall lab — completely different world.** 🤯

## 📚 What You'll Learn
- Images are just numbers (a preview of tomorrow's Computer Vision day! 👁️)
- Applying K-Means beyond spreadsheets
- Watching compression happen live on YOUR photo

📦 **Dataset:** your own photo! (A backup image is included if you don't have one.)

In [ ]:
!pip install numpy matplotlib scikit-learn

### 🛠️ Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

print("🎨 Brushes ready!")

### 📸 Load YOUR Photo

**Option A (recommended 😎):** upload a photo of yourself, your cat, your car — anything! Name it `my_photo.jpg` and place it next to this notebook. *(In Colab: click the 📁 folder icon on the left → upload.)*

**Option B:** no photo? No problem — we'll use a built-in flower image. 🌸

In [ ]:
try:
    img = plt.imread('my_photo.jpg')
    print("📸 Your photo loaded!")
except FileNotFoundError:
    from sklearn.datasets import load_sample_image
    img = load_sample_image('flower.jpg')
    print("🌸 No photo found — using the backup flower image.")

# Normalize pixel values to 0-1 (a common preprocessing step)
img = np.array(img, dtype=np.float64) / 255

plt.figure(figsize=(7, 5))
plt.imshow(img)
plt.axis('off')
plt.title('The original 🖼️')
plt.show()

print(f"📏 Image shape: {img.shape}  →  (height, width, 3 color channels: Red, Green, Blue)")

### 🔢 Section 1: An Image Is Just a Table of Numbers

Mind-blow incoming 🤯: every pixel is just 3 numbers — how much **Red**, **Green**, and **Blue** it contains (each from 0 to 1).

So we can **reshape** the image from a (height × width) grid into a simple table: one row per pixel, 3 columns (R, G, B). Sound familiar? **It's a dataset — exactly like the mall customers, but the "features" are colors!**

In [ ]:
h, w, c = img.shape
pixels = img.reshape(-1, 3)   # every row = one pixel = one "customer" 😄

print(f"🧮 Our 'dataset' has {pixels.shape[0]:,} pixels (rows) and {pixels.shape[1]} features (R, G, B)")
print(f"🌈 Number of DIFFERENT colors in the image: {len(np.unique(pixels, axis=0)):,}")
print("\nFirst 5 pixels (R, G, B):")
print(pixels[:5].round(3))

### 🤖 Section 2: Cluster the Colors!

Now the magic:
1. K-Means groups all pixels into **K color clusters**
2. Each cluster's **centroid** becomes the "official paint color" of that group 🎨
3. We repaint every pixel with its cluster's official color

Result: an image with only K colors!

*(Trick for speed: we train K-Means on a random sample of 10,000 pixels instead of millions — the color clusters are the same.)*

In [ ]:
def repaint(pixels, k, h, w):
    """Cluster pixel colors into k groups and repaint the image with the k centroid colors."""
    # Train on a random sample of pixels (for speed)
    rng = np.random.default_rng(42)
    sample = pixels[rng.choice(len(pixels), size=min(10000, len(pixels)), replace=False)]

    km = KMeans(n_clusters=k, random_state=42, n_init=4)
    km.fit(sample)

    # Assign EVERY pixel to its nearest color cluster, then repaint with the centroid color
    labels = km.predict(pixels)
    new_pixels = km.cluster_centers_[labels]
    return new_pixels.reshape(h, w, 3)

k = 8
compressed = repaint(pixels, k, h, w)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title('Original (thousands of colors)')
axes[1].imshow(compressed); axes[1].axis('off'); axes[1].set_title(f'K-Means repaint — only {k} colors! 🎨')
plt.show()

### 🌈 Section 3: The K Gallery — From Cartoon to Photo

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, k in zip(axes.flat, [2, 4, 8, 16]):
    ax.imshow(repaint(pixels, k, h, w))
    ax.axis('off')
    ax.set_title(f'K = {k} colors')

plt.suptitle('🎨 The same photo, painted with 2 → 16 colors', fontsize=14)
plt.tight_layout()
plt.show()

# 😲 Notice: at K=16 it already looks almost REAL — even though the original had thousands of colors!

### 🎯 Mini Exercise 3.1
What is the SMALLEST K where your photo still looks "acceptable" to you? Try K values between 8 and 32 and find your personal threshold. 👇

In [ ]:
my_k = 0   # TO-DO: try different values!

plt.figure(figsize=(7, 5))
plt.imshow(repaint(pixels, my_k, h, w))
plt.axis('off')
plt.title(f'My photo with K = {my_k} colors')
plt.show()

### 🎯 Mini Exercise 3.2 — The Paint Palette 🖌️
Let's actually SEE the K "official colors" the algorithm chose. Run the cell below with your favorite K. These centroids ARE the compression!

In [ ]:
k = 8   # TO-DO: try your own K

rng = np.random.default_rng(42)
sample = pixels[rng.choice(len(pixels), size=min(10000, len(pixels)), replace=False)]
km = KMeans(n_clusters=k, random_state=42, n_init=4).fit(sample)

palette = km.cluster_centers_.reshape(1, k, 3)
plt.figure(figsize=(10, 2))
plt.imshow(palette)
plt.axis('off')
plt.title(f'🎨 The {k} centroid colors K-Means chose for your photo')
plt.show()

## The GOLDEN Question 🏆

In the mall lab, a cluster was a *group of similar customers* and the centroid was the *typical customer* of that group.

**In THIS lab: what is a cluster? What is a centroid? And here's the big one — nobody ever labeled which pixels are "sky-blue" or "skin-tone", so why is this unsupervised learning and not classification?** 🤔

*Bonus: the original image stores 3 numbers per pixel. After compression with K=16, we only need to store the palette (16 colors) + one small number per pixel (which palette color it uses). Why does that make the file smaller?* 💾